# 18: One problem, three simulation representations

**For:** users who completed VQE in lab 08 and want to understand FlagQuantum's execution choices.

**Your mission:** use one circuit builder, one energy function, and one training loop with statevector, MPS, and tensor-network execution.
Before comparing training, check that all three compute the same energy and gradient at the same parameter values.

This is a correctness and API experiment. The two-qubit workload is intentionally small; it cannot establish a useful performance ranking.

## 1. Define the problem once

We minimize H = -Z₀Z₁ - 0.5X₀ - 0.5X₁. The dense 4×4 matrix gives an independent exact energy reference for this small system.
The circuit and observable below do not name a simulator.

In [ ]:
import torch
import flagquantum as fq
import matplotlib.pyplot as plt

observable = -1.0 * (fq.Z(0) @ fq.Z(1)) - 0.5 * fq.X(0) - 0.5 * fq.X(1)
Z = torch.tensor([[1.0, 0.0], [0.0, -1.0]], dtype=torch.float64)
X = torch.tensor([[0.0, 1.0], [1.0, 0.0]], dtype=torch.float64)
I = torch.eye(2, dtype=torch.float64)
H = -torch.kron(Z, Z) - 0.5 * torch.kron(X, I) - 0.5 * torch.kron(I, X)
exact_energy = torch.linalg.eigvalsh(H).min().item()


def circuit(theta):
    q = fq.Circuit(2)
    q.ry(0, theta[0])
    q.ry(1, theta[1])
    q.cx(0, 1)
    q.ry(0, theta[2])
    q.ry(1, theta[3])
    return q


print("Exact ground energy:", exact_energy)


## 2. Make representation an execution option

The mode is the only representation choice in this function. Disabling backend fallback makes an unsupported choice fail explicitly rather than silently substituting another implementation.
We request energy directly; no dense-state conversion is needed in the optimization loop.

In [ ]:
def evaluate(theta, mode):
    return fq.run(
        circuit(theta),
        options=fq.ExecutionOptions(
            mode=mode, device="cpu", allow_backend_fallback=False
        ),
        outputs=fq.expectation(observable),
    )


modes = ["statevector", "mps", "tensor_network"]
torch.manual_seed(7)
initial_theta = 0.2 * torch.randn(4)
probes = {}
for mode in modes:
    theta = initial_theta.clone().requires_grad_(True)
    result = evaluate(theta, mode)
    energy = result.expectation().sum()
    (gradient,) = torch.autograd.grad(energy, theta)
    probes[mode] = {
        "energy": energy.detach(),
        "gradient": gradient.detach(),
        "runtime": dict(result.runtime),
    }
    print(mode, "energy:", energy.item(), "runtime:", dict(result.runtime))


## 3. Check value and gradient agreement

Equal energies at one point are not enough for training: different gradients would drive the optimizer in different directions.
The tolerances below allow small floating-point differences in this CPU example.

In [ ]:
reference = probes["statevector"]
for mode in modes[1:]:
    torch.testing.assert_close(
        probes[mode]["energy"], reference["energy"], atol=1e-5, rtol=1e-5
    )
    torch.testing.assert_close(
        probes[mode]["gradient"], reference["gradient"], atol=1e-5, rtol=1e-5
    )
    print(
        mode,
        "maximum gradient difference:",
        (probes[mode]["gradient"] - reference["gradient"]).abs().max().item(),
    )


## 4. Reuse one training loop

Each run starts from the same parameters with a fresh optimizer. Do not reuse an optimizer's momentum state between representations.
We give every run the same 160-step budget.

In [ ]:
def train(mode):
    theta = torch.nn.Parameter(initial_theta.clone())
    optimizer = torch.optim.Adam([theta], lr=0.08)
    history = []
    for step in range(160):
        optimizer.zero_grad()
        energy = evaluate(theta, mode).expectation().sum()
        energy.backward()
        optimizer.step()
        history.append(energy.detach().item())
    final_energy = evaluate(theta, mode).expectation().item()
    return {
        "history": history,
        "final_energy": final_energy,
        "parameters": theta.detach().tolist(),
    }


runs = {mode: train(mode) for mode in modes}
for mode, run in runs.items():
    gap = run["final_energy"] - exact_energy
    print(mode, "final energy:", run["final_energy"], "gap:", gap)
    assert abs(gap) < 0.02


## 5. Compare the learning curves

The curves should approach the same energy. Small differences are normal because the representations can follow different floating-point operation orders.
We have demonstrated a shared program and objective, not that every representation has the same speed or memory cost.

In [ ]:
for mode, run in runs.items():
    plt.plot(run["history"], label=mode)
plt.axhline(exact_energy, color="black", linestyle="--", label="Exact ground energy")
plt.xlabel("Optimizer step")
plt.ylabel("Energy")
plt.legend()
plt.show()
assert (
    max(r["final_energy"] for r in runs.values())
    - min(r["final_energy"] for r in runs.values())
    < 2e-4
)


## 6. Save the comparison

In [ ]:
from pathlib import Path

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "flagquantum").is_dir()
)
OUTPUTS = ROOT / "workshops/flagos2026/outputs"
OUTPUTS.mkdir(exist_ok=True)


In [ ]:
import json

report = {
    "experiment": "one_objective_three_representations",
    "source": "local_simulation",
    "flagquantum_version": fq.__version__,
    "torch_version": torch.__version__,
    "seed": 7,
    "steps": 160,
    "learning_rate": 0.08,
    "qubits": 2,
    "initial_parameters": initial_theta.tolist(),
    "exact_energy": exact_energy,
    "final_energies": {mode: run["final_energy"] for mode, run in runs.items()},
    "fallback_allowed": False,
    "scope": "CPU correctness and gradient parity; no performance or scalability claim",
}
(OUTPUTS / "three_simulators_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))


## Explain it to someone else

- Which lines describe the scientific problem, and which select the execution representation?
- What did the gradient comparison establish that the final energy comparison did not?
- Why would increasing qubit count require a new experiment rather than extrapolating from these timings?

**Try next:** change the field strength consistently in both the observable and the reference matrix. Repeat the parity checks before training.
Then use [lab 11](11_mps.ipynb) to investigate approximation and storage with explicit bond limits.
Record your observations in [Field notes](../../FIELD_NOTES.md).